# PathSnap — Notebook 01: Dataset exploration, candidate generation and model comparison (Batch 1)
UE24CS352A ML Hackathon. **Colab:** Runtime → Run all. When asked, upload `pathsnap_colab_bundle.zip` (holds `src/` + `Student folder/`).
Partitions used: **train (80 trips)** for EDA statistics, fitting and GroupKFold CV; **student_test (10 trips)** only for model comparison. Evaluator trips are never touched.

In [ ]:
import os, sys, glob, zipfile, time, json, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install shapely pyproj folium lightgbm optuna
    if not os.path.isdir('/content/PathSnap/src'):
        from google.colab import files
        up = files.upload()                      # choose pathsnap_colab_bundle.zip
        zipfile.ZipFile(list(up)[0]).extractall('/content/PathSnap')
    BASE = '/content/PathSnap'
else:
    BASE = os.environ.get('PATHSNAP_BASE', os.path.abspath('..'))
DATA = os.environ.get('PATHSNAP_DATA', os.path.join(BASE, 'Student folder'))
sys.path.insert(0, os.path.join(BASE, 'src'))
RES = os.path.join(BASE, 'results'); PLOTS = os.path.join(RES, 'plots'); MAPS = os.path.join(RES, 'maps')
for d in (RES, PLOTS, MAPS): os.makedirs(d, exist_ok=True)
print('BASE', BASE, '| DATA', DATA)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathsnap.io import list_trips, load_trip, clean_track
from pathsnap.network import RoadNetwork
from pathsnap.pipeline import prepare_trip, DEFAULT_CFG
from pathsnap.features import FEATURES
from pathsnap.metrics import candidate_miss_rate, route_continuity, pointwise_accuracy
from pathsnap.viz import trip_map
SEED = 42; np.random.seed(SEED)
TRAIN = os.path.join(DATA, 'train'); STEST = os.path.join(DATA, 'student_test')
train_ids, stest_ids = list_trips(TRAIN), list_trips(STEST)
print('train trips:', len(train_ids), '| student-test trips:', len(stest_ids))
print('config:', DEFAULT_CFG)

## 1. Dataset inventory and data quality (both partitions)

In [ ]:
rows = []
for part, root, ids in [('train', TRAIN, train_ids), ('student_test', STEST, stest_ids)]:
    for tid in ids:
        tr = load_trip(root, tid)
        clean, rep = clean_track(tr.track)
        dt = np.diff(clean[:, 2])
        net = RoadNetwork(tr.nodes, tr.arcs)
        comp = net.components()
        sizes = {e: os.path.getsize(os.path.join(root, tid, f'{tid}.{e}')) for e in ['track','nodes','arcs','route']}
        rows.append(dict(partition=part, trip=tid, gps_points=len(tr.track), **{k: v for k, v in rep.items() if k != 'raw_points'},
            nan_values=int(np.isnan(tr.track).sum()), nodes=len(tr.nodes), arcs=len(tr.arcs), route_arcs=len(tr.route),
            route_arcs_valid=bool((tr.route < len(tr.arcs)).all()),
            dt_1s_frac=float((dt == 1).mean()) if len(dt) else np.nan, max_gap_s=float(dt.max()) if len(dt) else 0,
            gaps_over_5s=int((dt > 5).sum()), duration_s=float(clean[-1, 2] - clean[0, 2]),
            lon_min=clean[:,0].min(), lon_max=clean[:,0].max(), lat_min=clean[:,1].min(), lat_max=clean[:,1].max(),
            utm_epsg=net.epsg, **comp, **{f'size_{k}_kb': v/1024 for k, v in sizes.items()}))
inv = pd.DataFrame(rows); inv.to_csv(os.path.join(RES, 'eda_inventory.csv'), index=False)
print(inv.groupby('partition')[['gps_points','nodes','arcs','route_arcs']].agg(['count','sum','mean','min','max']).T)
inv.head()

In [ ]:
print('Missing / invalid values (all trips):')
print(inv[['invalid_rows','nan_values','out_of_order','duplicate_timestamps']].sum())
print('\nAll route arc ids valid:', inv.route_arcs_valid.all())
print('Share of 1-second steps (mean over trips): %.3f' % inv.dt_1s_frac.mean())
print('\nTrips with large GPS gaps (>5 s):'); print(inv[inv.gaps_over_5s > 0][['partition','trip','max_gap_s','gaps_over_5s']])
print('\nRoad-network connectivity summary:'); print(inv[['weak_components','strong_components','largest_scc_frac','isolated_nodes']].describe().T)
print('\nUTM zones used:', sorted(inv.utm_epsg.unique()))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
t = inv[inv.partition=='train']
ax[0,0].hist(t.gps_points, bins=30, color='#1f77b4'); ax[0,0].set_title('GPS points per trip (train)')
ax[0,1].scatter(t.nodes, t.arcs, s=12); ax[0,1].set_xscale('log'); ax[0,1].set_yscale('log'); ax[0,1].set_title('Nodes vs arcs per trip (train)'); ax[0,1].set_xlabel('nodes'); ax[0,1].set_ylabel('arcs')
ax[0,2].hist(t.duration_s/60, bins=30, color='#2ca02c'); ax[0,2].set_title('Trip duration (min, train)')
alldt = np.concatenate([np.diff(clean_track(load_trip(TRAIN, i).track)[0][:,2]) for i in train_ids])
vals, cnts = np.unique(np.minimum(alldt, 20), return_counts=True)
ax[1,0].bar(vals, cnts, color='#ff7f0e'); ax[1,0].set_yscale('log'); ax[1,0].set_title('Sampling interval dt (s, capped 20, train)')
ax[1,1].hist(t.strong_components, bins=30, color='#9467bd'); ax[1,1].set_title('Strongly connected components (train)')
ax[1,2].scatter(inv.lon_min, inv.lat_min, c=(inv.partition=='train'), cmap='coolwarm', s=15); ax[1,2].set_title('Trip locations (lon, lat), both partitions')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'eda_overview.png'), dpi=120); plt.show()

In [ ]:
# GPS trajectory + road network of one training trip (projected metres)
from pathsnap.geo import project
tid = train_ids[0]; tr = load_trip(TRAIN, tid); net = RoadNetwork(tr.nodes, tr.arcs)
pxy = project(clean_track(tr.track)[0][:, :2], net.epsg)
fig, ax = plt.subplots(figsize=(8, 8))
pad = 300; x0, x1, y0, y1 = pxy[:,0].min()-pad, pxy[:,0].max()+pad, pxy[:,1].min()-pad, pxy[:,1].max()+pad
sel = np.where((net.ax > x0) & (net.ax < x1) & (net.ay > y0) & (net.ay < y1))[0]
for a in sel: ax.plot([net.ax[a], net.bx[a]], [net.ay[a], net.by[a]], color='#bbb', lw=0.6)
for a in tr.route: ax.plot([net.ax[a], net.bx[a]], [net.ay[a], net.by[a]], color='green', lw=2.5)
ax.scatter(pxy[:,0], pxy[:,1], s=3, c='#1f4fd1', label='GPS'); ax.set_aspect('equal'); ax.set_title(f'Trip {tid}: GPS (blue), reference route (green), network (grey) — UTM EPSG:{net.epsg}')
plt.savefig(os.path.join(PLOTS, 'trajectory_example.png'), dpi=120); plt.show()

In [ ]:
# Interactive Folium map (raw GPS, road network, reference route) — train trip
m = trip_map(clean_track(tr.track)[0], net, ref_route=tr.route)
m.save(os.path.join(MAPS, f'eda_trip_{tid}.html')); m

## 2. Per-point labels, candidate generation and features
The `.route` lists arcs, not one label per GPS point, so labels are derived by **monotonic DP alignment** (`pathsnap/labels.py`). Candidates = arcs within R m, at most K nearest (`pathsnap/candidates.py`).

In [ ]:
t0 = time.time(); prepared = {}
for part, root, ids in [('train', TRAIN, train_ids), ('student_test', STEST, stest_ids)]:
    for tid in ids:
        prepared[(part, tid)] = prepare_trip(load_trip(root, tid))
print('prepared %d trips in %.1fs' % (len(prepared), time.time() - t0))
def frame(part):
    out = []
    for (p, tid), pt in prepared.items():
        if p == part:
            d = pt.df.copy(); d['gpt'] = tid + '_' + d['pt'].astype(str); out.append(d)
    return pd.concat(out, ignore_index=True)
TR, TE = frame('train'), frame('student_test')
stats = []
for (p, tid), pt in prepared.items():
    stats.append(dict(partition=p, trip=tid, points=len(pt.pxy), rows=len(pt.df), miss_rate=candidate_miss_rate(pt.df, pt.ref),
                      no_cand_pts=len(pt.pxy) - pt.df.pt.nunique(), label_dist_p50=np.median(pt.ref_dist), label_dist_p95=np.percentile(pt.ref_dist, 95)))
cs = pd.DataFrame(stats); cs.to_csv(os.path.join(RES, 'candidate_stats.csv'), index=False)
print(cs.groupby('partition')[['points','rows','miss_rate','no_cand_pts','label_dist_p50','label_dist_p95']].mean())
print('rows: train', len(TR), '| student-test', len(TE), '| positive share train %.3f' % TR.label.mean())
print('Candidate miss rate (point-weighted): train %.4f | student-test %.4f' % (
      np.average(cs[cs.partition=='train'].miss_rate, weights=cs[cs.partition=='train'].points),
      np.average(cs[cs.partition=='student_test'].miss_rate, weights=cs[cs.partition=='student_test'].points)))

In [ ]:
alld = np.concatenate([pt.ref_dist for (p, _), pt in prepared.items() if p == 'train'])
print('distance GPS -> label arc percentiles 50/90/95/99:', np.round(np.percentile(alld, [50, 90, 95, 99]), 1))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(np.minimum(alld, 60), bins=60); ax[0].axvline(DEFAULT_CFG['radius'], c='r', ls='--', label='radius R'); ax[0].legend(); ax[0].set_title('GPS-to-reference-arc distance (m, train)')
r = TR[TR.label == 1].dist_rank.value_counts().sort_index(); ax[1].bar(r.index, r.values); ax[1].set_title('Distance rank of the true candidate (train)')
TR.groupby('label')[['dist','win_dist','heading_diff']].mean().T.plot.bar(ax=ax[2]); ax[2].set_title('Mean feature value by label (train)')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'labels_candidates.png'), dpi=120); plt.show()
TR[FEATURES].describe().T

## 3. Model experimentation (7 models, 7 families)
Binary task: is a candidate the reference arc? Top-1 = highest-probability candidate per GPS point (**primary metric**).
**GroupKFold (5 folds, grouped by trip)** on the 80 train trips. To keep Colab time reasonable every model is fitted on the same random sample of training GPS points (all candidates of each sampled point), and CV folds are scored on a sample of validation points. Same preprocessing for all models.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, log_loss, confusion_matrix
import lightgbm as lgb
pos_w = (1 - TR.label.mean()) / TR.label.mean()
MODELS = {
 'LogisticRegression': ('Linear', lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight='balanced', max_iter=500)), 'C=1.0, balanced, scaled'),
 'GaussianNB':        ('Probabilistic', lambda: make_pipeline(StandardScaler(), GaussianNB()), 'default var_smoothing'),
 'KNN':               ('Instance-based', lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25, weights='distance', n_jobs=-1)), 'k=25, distance-weighted, scaled'),
 'RandomForest':      ('Ensemble (bagging)', lambda: RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED), '150 trees, depth 14, leaf 5'),
 'ExtraTrees':        ('Ensemble (randomised trees)', lambda: ExtraTreesClassifier(n_estimators=200, max_depth=16, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=SEED), '200 trees, depth 16, leaf 3'),
 'LightGBM':          ('Boosting', lambda: lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, scale_pos_weight=pos_w, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1), '400 trees, lr 0.05, 31 leaves'),
 'MLP':               ('Neural network', lambda: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, batch_size=512, max_iter=40, early_stopping=True, random_state=SEED)), '64-32 ReLU, adam, early stop'),
}
def sample_points(df, n, seed=SEED):
    g = df.gpt.unique(); rng = np.random.default_rng(seed)
    keep = set(rng.choice(g, size=min(n, len(g)), replace=False)); return df[df.gpt.isin(keep)]
def top1(df, p):
    d = df[['gpt', 'label']].assign(p=p); idx = d.groupby('gpt')['p'].idxmax()
    return float(d.loc[idx, 'label'].mean())   # points with no candidate are excluded here; counted in miss rate
def scores(df, p):
    yh = (p >= 0.5).astype(int); y = df.label.values
    return dict(acc=accuracy_score(y, yh), precision=precision_score(y, yh, average='macro', zero_division=0),
                recall=recall_score(y, yh, average='macro'), macro_f1=f1_score(y, yh, average='macro'),
                log_loss=log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)), top1=top1(df, p))
N_FIT, N_VAL = 25000, 8000
print('positive class weight %.2f' % pos_w)

In [ ]:
gkf = GroupKFold(n_splits=5); cv_rows = []
for name, (fam, mk, cfg) in MODELS.items():
    for fold, (tri, vai) in enumerate(gkf.split(TR, groups=TR.trip)):
        ftr = sample_points(TR.iloc[tri], N_FIT, SEED + fold); fva = sample_points(TR.iloc[vai], N_VAL, SEED + fold)
        m = mk(); t0 = time.time(); m.fit(ftr[FEATURES], ftr.label); tt = time.time() - t0
        p = m.predict_proba(fva[FEATURES])[:, 1]; ptr = m.predict_proba(ftr[FEATURES])[:, 1]
        s = scores(fva, p); cv_rows.append(dict(model=name, fold=fold, train_macro_f1=scores(ftr, ptr)['macro_f1'], train_top1=top1(ftr, ptr), fit_s=tt, **s))
    r = pd.DataFrame([x for x in cv_rows if x['model'] == name])
    print(f"{name:18s} CV macroF1 {r.macro_f1.mean():.4f}±{r.macro_f1.std():.4f}  top1 {r.top1.mean():.4f}  logloss {r.log_loss.mean():.4f}  train-F1 {r.train_macro_f1.mean():.4f}")
cv = pd.DataFrame(cv_rows); cv.to_csv(os.path.join(RES, 'cv_folds.csv'), index=False)

In [ ]:
# Final fit of every model on the training sample, evaluation on the 10 student-test trips
from pathsnap.hmm import precompute_steps, classifier_top1, per_point
fit_df = sample_points(TR, 2 * N_FIT, SEED); final_rows = []; fitted = {}; probs_te = {}
steps_te = {tid: precompute_steps(prepared[('student_test', tid)].df, prepared[('student_test', tid)].pxy, prepared[('student_test', tid)].times, prepared[('student_test', tid)].net) for tid in stest_ids}
for name, (fam, mk, cfg) in MODELS.items():
    m = mk(); t0 = time.time(); m.fit(fit_df[FEATURES], fit_df.label); tt = time.time() - t0
    t0 = time.time(); p = m.predict_proba(TE[FEATURES])[:, 1]; it = time.time() - t0
    fitted[name] = m; probs_te[name] = p; s = scores(TE, p)
    acc_pts, cont = [], []
    for tid in stest_ids:
        pt = prepared[('student_test', tid)]; pp = p[(TE.trip == tid).values]
        pred = per_point(steps_te[tid], classifier_top1(steps_te[tid], pp), len(pt.pxy))
        acc_pts.append(pointwise_accuracy(pred, pt.ref)); cont.append(route_continuity(pred, pt.net))
    c = cv[cv.model == name]
    final_rows.append(dict(model=name, family=fam, config=cfg, cv_macro_f1=c.macro_f1.mean(), cv_macro_f1_std=c.macro_f1.std(), cv_top1=c.top1.mean(), cv_log_loss=c.log_loss.mean(),
        test_accuracy=s['acc'], test_precision=s['precision'], test_recall=s['recall'], test_macro_f1=s['macro_f1'], test_log_loss=s['log_loss'],
        test_top1_obs=float(np.mean(acc_pts)), test_route_continuity=float(np.mean(cont)), train_time_s=tt, infer_time_s=it, infer_ms_per_1k_rows=1000 * it / len(TE) * 1000))
    print(f"{name:18s} test top1(all points) {np.mean(acc_pts):.4f}  macroF1 {s['macro_f1']:.4f}  logloss {s['log_loss']:.4f}  continuity {np.mean(cont):.4f}  fit {tt:.1f}s")
comp = pd.DataFrame(final_rows).sort_values('test_top1_obs', ascending=False); comp.to_csv(os.path.join(RES, 'model_comparison.csv'), index=False)
comp.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5)); order = comp.model.tolist()
cv.boxplot(column='macro_f1', by='model', ax=ax[0], rot=45); ax[0].set_title('CV macro F1 per fold (train, GroupKFold)'); ax[0].set_xlabel('')
x = np.arange(len(order)); ax[1].bar(x - 0.2, comp.test_top1_obs, 0.4, label='Top-1 accuracy'); ax[1].bar(x + 0.2, comp.test_macro_f1, 0.4, label='Macro F1')
ax[1].set_xticks(x); ax[1].set_xticklabels(order, rotation=45); ax[1].set_ylim(0.5, 1); ax[1].legend(); ax[1].set_title('Student-test Top-1 and macro F1')
ax[2].bar(order, comp.test_log_loss, color='#d62728'); ax[2].set_xticklabels(order, rotation=45); ax[2].set_title('Student-test log loss (lower is better)')
plt.suptitle(''); plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'model_comparison.png'), dpi=120); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(order, comp.test_route_continuity, color='#2ca02c'); ax[0].set_ylim(0.8, 1); ax[0].set_xticklabels(order, rotation=45); ax[0].set_title('Route continuity (classifier-only, student-test)')
ax[1].bar(order, comp.infer_ms_per_1k_rows, color='#7f7f7f'); ax[1].set_yscale('log'); ax[1].set_xticklabels(order, rotation=45); ax[1].set_title('Inference time (ms per 1k rows, log)')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'continuity_speed.png'), dpi=120); plt.show()

In [ ]:
# Confusion matrices (student-test, threshold 0.5)
fig, ax = plt.subplots(1, len(order), figsize=(3.2 * len(order), 3))
for a, name in zip(ax, order):
    cm = confusion_matrix(TE.label, (probs_te[name] >= 0.5).astype(int), normalize='true')
    a.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2): a.text(j, i, f'{cm[i,j]:.2f}', ha='center', va='center')
    a.set_title(name, fontsize=9); a.set_xticks([0,1]); a.set_yticks([0,1]); a.set_xlabel('pred'); a.set_ylabel('true')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'confusion_matrices.png'), dpi=120); plt.show()

In [ ]:
# Training histories — only for models that really have one (MLP loss curve, LightGBM eval log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
mlp = fitted['MLP'][-1]; ax[0].plot(mlp.loss_curve_, label='training loss'); ax[0].plot(1 - np.array(mlp.validation_scores_), label='1 - validation accuracy (10% hold-out)'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].set_title('MLP training history')
va_trips = set(train_ids[::5]); a_ = sample_points(TR[~TR.trip.isin(va_trips)], 2 * N_FIT); b_ = sample_points(TR[TR.trip.isin(va_trips)], N_VAL)
g = MODELS['LightGBM'][1](); g.fit(a_[FEATURES], a_.label, eval_set=[(a_[FEATURES], a_.label), (b_[FEATURES], b_.label)], eval_names=['train', 'valid'], eval_metric='binary_logloss')
for k in ['train', 'valid']: ax[1].plot(g.evals_result_[k]['binary_logloss'], label=k)
ax[1].set_xlabel('boosting round'); ax[1].legend(); ax[1].set_title('LightGBM log loss per round (validation = 16 held-out train trips)')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'training_histories.png'), dpi=120); plt.show()
imp = pd.Series(fitted['LightGBM'].feature_importances_, index=FEATURES).sort_values(); imp.plot.barh(figsize=(6, 5), title='LightGBM feature importance (splits)'); plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'feature_importance.png'), dpi=120); plt.show()

In [ ]:
# Folium: classifier-only route of the best model on a student-test trip
from pathsnap.hmm import reconstruct_route
best = comp.iloc[0].model; tid = stest_ids[0]; pt = prepared[('student_test', tid)]; trt = load_trip(STEST, tid)
pred = per_point(steps_te[tid], classifier_top1(steps_te[tid], probs_te[best][(TE.trip == tid).values]), len(pt.pxy))
m = trip_map(pt.track, pt.net, ref_route=trt.route, routes={f'{best} classifier-only route': reconstruct_route(pred, pt.net, fill=False)})
m.save(os.path.join(MAPS, f'classifier_only_{best}_{tid}.html')); print('best model on student-test Top-1:', best); m

In [ ]:
# Save everything for Notebook 02 / report
import joblib, platform, sklearn
os.makedirs(os.path.join(BASE, 'artifacts'), exist_ok=True)
TR.to_parquet(os.path.join(RES, 'features_train.parquet')); TE.to_parquet(os.path.join(RES, 'features_student_test.parquet'))
json.dump({'python': platform.python_version(), 'sklearn': sklearn.__version__, 'lightgbm': lgb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
           'cfg': DEFAULT_CFG, 'features': FEATURES, 'N_FIT': N_FIT, 'N_VAL': N_VAL, 'seed': SEED}, open(os.path.join(RES, 'run_config_nb01.json'), 'w'), indent=1)
if IN_COLAB:
    import shutil; shutil.make_archive('/content/pathsnap_results_nb01', 'zip', BASE, 'results')
    from google.colab import files; files.download('/content/pathsnap_results_nb01.zip')
print('saved to', RES)